In [1]:
import sys
from pathlib import Path

# 1. Add the parent directory (src/) to sys.path
src_dir = Path.cwd().parent
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# 2. Use absolute imports (no leading dots)
from dotenv import load_dotenv
from ollama import ChatResponse

from helpers.config import get_settings
from stores.llm.LLMEnums import LLMEnums
from stores.llm.LLMProviderFactory import LLMProviderFactory

# 3. Run your test
load_dotenv()
config = get_settings()
factory = LLMProviderFactory(config=config)
client = factory.create(LLMEnums.OLLAMA.value)

client.set_generation_model(config.GENERATION_MODEL_ID)
response: ChatResponse = client.generate_response("Hi", None)

print(response.message.content)

Hi there! I'm GLM, trained by Z.ai. How can I assist you today? Whether you have questions, need information, or just want to chat, I'm here to help.


In [2]:
from stores.vectordb.VectorDBProviderFactory import VectorDBProviderFactory
from stores.vectordb.VectorDBEnums import VectorDBEnums

In [3]:
factory_db = VectorDBProviderFactory(config)

In [4]:
client_db = factory_db.create(provider=VectorDBEnums.QDRANT.value)

In [5]:
client_db.connect()

In [6]:
client_embed=factory.create(LLMEnums.OLLAMAE.value)

In [7]:
client_embed.set_embedding_model(config.EMBEDDING_MODEL_ID,config.EMBEDDING_MODEL_SIZE)

In [8]:
from tools.search_doctors import DoctorTools

doc_tool = DoctorTools(client_embed,client_db)
tool = doc_tool.search_doctors

In [9]:
from prompts.prompt_templatet import format_prompt,SYSTEM_PROMPT

In [10]:
query = "My leg is broken"

In [11]:
messages = [
    {
        "role":"system",
        "content":SYSTEM_PROMPT
    }
]

In [12]:
response =client.generate_response(prompt=query,
                                   chat_history=messages,
                                   tools=[tool])

In [13]:
response

ChatResponse(model='glm-5.2:cloud', created_at='2026-08-23T12:46:39.226284886Z', done=True, done_reason='stop', total_duration=5618380778, load_duration=None, prompt_eval_count=1056, prompt_eval_duration=None, eval_count=411, eval_duration=None, message=Message(role='assistant', content="I'm so sorry to hear about your leg — that sounds very painful and stressful. Let me help you right away.\n\n---\n\n### 🚨 First — Emergency Check\n\nA broken leg can be serious. **If you experience any of the following, please call 123 (Ambulance in Egypt) or go to the nearest Emergency Room immediately:**\n\n- Bone visible through the skin (open fracture)\n- Severe deformity or the leg is bent at an unnatural angle\n- Numbness, tingling, or loss of feeling in the leg or foot\n- The leg or foot looks pale, blue, or cold\n- Heavy bleeding\n- You are unable to move or bear any weight at all\n\n**If none of the above apply and the fracture has been stabilized or previously treated**, you'll need to follow

In [14]:
response.message.tool_calls

In [15]:
from agent.agent import Agent

agent = Agent(client,[tool],client_embedding=client_embed,
              client_vectorDB=client_db)

In [16]:
f,m,c=agent.run("my leg is broken")

In [17]:
print(f)

I'm sorry to hear about your leg — that sounds very painful and stressful. Let me help you right away. 🚨

---

### ⚠️ Important: Is This an Emergency?

A broken leg can be a **medical emergency**, especially if you notice any of the following red flags:

- **Severe pain** that doesn't improve
- **Visible deformity** or bone protruding through the skin
- **Numbness, tingling, or loss of sensation** in the leg or foot
- **Inability to bear any weight** or move the leg
- **Swelling, bruising, or cold/pale skin** below the injury
- **Heavy bleeding**

> 🚑 **If any of the above apply, please call 123 (Ambulance in Egypt) immediately or go to the nearest Emergency Room (ER).** Do not attempt to move the leg unnecessarily — keep it immobilized and elevated if possible.

---

### 🦴 If You Need an Orthopedic Specialist (Follow-up / Non-Emergency)

If you've already received emergency care or this is a follow-up, I can help you find an **Orthopedic (عظام) doctor** near you. To give you the best 

In [18]:
m

[{'role': 'system',
  'content': 'You are "Tabeeby" (طبيبي), an intelligent, empathetic, and professional AI Medical Assistant and Healthcare Navigator powered by the Vezeeta doctor network in Egypt.\n\n### YOUR ROLE & OBJECTIVES:\n1. **Medical Orientation & Guidance**: Listen carefully to the patient\'s symptoms or health inquiries, provide empathetic and evidence-based health information, and help determine the most suitable medical specialty (e.g., Cardiology, Dermatology, Orthopedics, Pediatrics, ENT, Internal Medicine, etc.).\n2. **Doctor Discovery & Recommendation**: Recommend the best-matching qualified doctors from your database/context based on the patient\'s symptoms, required specialty, location/area (e.g., Nasr City, Maadi, Dokki, Mohandessin, Heliopolis, Alexandria, etc.), consultation fee budget, and patient ratings.\n3. **Clear Next Steps**: Provide practical next steps, questions to ask the doctor, or general lifestyle/supportive measures.\n\n### CRITICAL MEDICAL GUARDR

In [19]:
c

0.0040506

In [20]:
f,m,c = agent.run("can you search for me a doc in cairo and price be less than 400  that help me",
                  chat_history=m)

In [21]:
print(f)

Great news! I found several orthopedic doctors in Cairo for your broken leg. While no doctors were found strictly **under** 400 EGP, I found several excellent options at **exactly 400 EGP** — just at your budget limit. Here are the best matches:

---

### 🦴 Recommended Orthopedic Doctors (400 EGP)

#### 1. Dr. Tamer Batea ⭐ (Top Pick — Most Reviewed!)
- **Specialty**: Orthopedics — Arthroscopy & Sports Injuries, Foot & Ankle Orthopedics, Joint Replacement, Adult Orthopedics, Hand & Upper Limb Surgery, Adult & Pediatric Orthopedic Surgery
- **Clinic Location**: Nasr City
- **Consultation Fee**: 400 EGP
- **Rating & Reviews**: 663 reviews | ⏱️ Waiting time ~24 mins
- **Profile & Booking**: [Book with Dr. Tamer Batea](https://www.vezeeta.com/en/dr/Doctor-Tamer-Batea-Orthopedics)
- **Why this doctor**: Highly experienced with a large number of patient reviews and a broad range of orthopedic subspecialties, including foot and ankle care — ideal for a broken leg.

---

#### 2. Dr. Mohamed He

In [22]:
m

[{'role': 'system',
  'content': 'You are "Tabeeby" (طبيبي), an intelligent, empathetic, and professional AI Medical Assistant and Healthcare Navigator powered by the Vezeeta doctor network in Egypt.\n\n### YOUR ROLE & OBJECTIVES:\n1. **Medical Orientation & Guidance**: Listen carefully to the patient\'s symptoms or health inquiries, provide empathetic and evidence-based health information, and help determine the most suitable medical specialty (e.g., Cardiology, Dermatology, Orthopedics, Pediatrics, ENT, Internal Medicine, etc.).\n2. **Doctor Discovery & Recommendation**: Recommend the best-matching qualified doctors from your database/context based on the patient\'s symptoms, required specialty, location/area (e.g., Nasr City, Maadi, Dokki, Mohandessin, Heliopolis, Alexandria, etc.), consultation fee budget, and patient ratings.\n3. **Clear Next Steps**: Provide practical next steps, questions to ask the doctor, or general lifestyle/supportive measures.\n\n### CRITICAL MEDICAL GUARDR

In [23]:
c

0.0234988

In [24]:
f,m,c = agent.run("can you search for me a doc in new cairo and price be less than 400  that help me",
                  chat_history=m)

In [25]:
print(f)

Great news! I found **4 orthopedic doctors** in **New Cairo** with consultation fees **under 400 EGP** — all within your budget! 🎉

---

### 🦴 Recommended Orthopedic Doctors in New Cairo

---

#### 1. Dr. AbdElsamie Halawa ⭐ (Top Pick — Most Reviewed!)
- **Specialty**: Orthopedics — Adult Orthopedics, Arthroscopy & Sports Injuries, Foot & Ankle Orthopedics, Joint Replacement, Adult Orthopedic Surgery, Pediatric Orthopedic Surgery
- **Clinic Location**: New Cairo
- **Consultation Fee**: 300 EGP
- **Rating & Reviews**: 13 reviews | ⏱️ Waiting time ~23 mins
- **Profile & Booking**: [Book with Dr. AbdElsamie Halawa](https://www.vezeeta.com/en/dr/doctor-abdelsamie-halawa-orthopedics-3)
- **Why this doctor**: Has the most patient reviews among the results, with expertise in foot & ankle orthopedics and adult orthopedic surgery — directly relevant to a broken leg.

---

#### 2. Dr. Ahmed Esmat
- **Specialty**: Orthopedics — Arthroscopy & Sports Injuries, Adult Spine Surgery, Bone Deformities,

In [26]:
m

[{'role': 'system',
  'content': 'You are "Tabeeby" (طبيبي), an intelligent, empathetic, and professional AI Medical Assistant and Healthcare Navigator powered by the Vezeeta doctor network in Egypt.\n\n### YOUR ROLE & OBJECTIVES:\n1. **Medical Orientation & Guidance**: Listen carefully to the patient\'s symptoms or health inquiries, provide empathetic and evidence-based health information, and help determine the most suitable medical specialty (e.g., Cardiology, Dermatology, Orthopedics, Pediatrics, ENT, Internal Medicine, etc.).\n2. **Doctor Discovery & Recommendation**: Recommend the best-matching qualified doctors from your database/context based on the patient\'s symptoms, required specialty, location/area (e.g., Nasr City, Maadi, Dokki, Mohandessin, Heliopolis, Alexandria, etc.), consultation fee budget, and patient ratings.\n3. **Clear Next Steps**: Provide practical next steps, questions to ask the doctor, or general lifestyle/supportive measures.\n\n### CRITICAL MEDICAL GUARDR

In [27]:
c

0.048854

In [28]:
f,m,c = agent.run("less than 400 and not eqal 400 also",
                  chat_history=m)

In [29]:
print(f)

No problem at all! The good news is that **all the doctors I found in New Cairo are already strictly under 400 EGP** — none of them are at 400! 🎉

Here's a quick recap of your options, all **under 400 EGP**:

---

### 🦴 Orthopedic Doctors in New Cairo (All Under 400 EGP)

| # | Doctor | Fee | Reviews | Location |
|---|--------|-----|---------|----------|
| 1 | **Dr. AbdElsamie Halawa** ⭐ | **300 EGP** | 13 reviews | New Cairo |
| 2 | **Dr. Ahmed Esmat** | **300 EGP** | New profile | New Cairo |
| 3 | **Dr. Mohamed Hosam Mehrz** | **300 EGP** | New profile | New Cairo |
| 4 | **Dr. Ahmed Mohamed Abdelaziz** | **350 EGP** | New profile | New Cairo |

---

### 🏆 My Top Recommendation:

**Dr. AbdElsamie Halawa** — 300 EGP
- The only doctor with patient reviews (13 reviews)
- Specializes in **Foot & Ankle Orthopedics**, **Adult Orthopedic Surgery**, and **Joint Replacement**
- Waiting time: ~23 mins
- 📅 [Book your appointment here](https://www.vezeeta.com/en/dr/doctor-abdelsamie-halawa-orth

In [30]:
m

[{'role': 'system',
  'content': 'You are "Tabeeby" (طبيبي), an intelligent, empathetic, and professional AI Medical Assistant and Healthcare Navigator powered by the Vezeeta doctor network in Egypt.\n\n### YOUR ROLE & OBJECTIVES:\n1. **Medical Orientation & Guidance**: Listen carefully to the patient\'s symptoms or health inquiries, provide empathetic and evidence-based health information, and help determine the most suitable medical specialty (e.g., Cardiology, Dermatology, Orthopedics, Pediatrics, ENT, Internal Medicine, etc.).\n2. **Doctor Discovery & Recommendation**: Recommend the best-matching qualified doctors from your database/context based on the patient\'s symptoms, required specialty, location/area (e.g., Nasr City, Maadi, Dokki, Mohandessin, Heliopolis, Alexandria, etc.), consultation fee budget, and patient ratings.\n3. **Clear Next Steps**: Provide practical next steps, questions to ask the doctor, or general lifestyle/supportive measures.\n\n### CRITICAL MEDICAL GUARDR

In [31]:
c

0.0645242